<a href="https://colab.research.google.com/github/catarinayee/MALDI-Assistant/blob/main/maldiast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Purpose**: This block of code installs the necessary packages for this program to run

**Input**: None :)

In [ ]:
# Install what we need.
!pip install -q numpy pandas

[**Optional**]

**Purpose**: The next block of code allows you to **merge two spreadsheets** together. Skip this if you only have one file.

**Input**: Upload **Two(2) METASPACE .CSV files** here that you just downloaded, **one at a time**.

In [ ]:
import sys
import os
import traceback
import numpy as np
import pandas as pd
from google.colab import files

# Stores the uploaded files and their names in a variable
mergefile1 = files.upload()
mergefile2 = files.upload()
mf1name = list(mergefile1.keys())[0]
mf2name = list(mergefile2.keys())[0]

# Reads the files
mf1df = pd.read_csv(mf1name, skiprows=2)
mf2df = pd.read_csv(mf2name, skiprows=2)

# Merges the files vertically
mergedfile = pd.concat([mf1df, mf2df])

# Sorts the file by m.z value, increasing
mergedfile.sort_values(by="mz")

# Exports the file in CSV format
mergedfile.to_csv("Merged.csv", index=False)

**Purpose**: This block of code will clean up your METASPACE file and give you a list of mz that you can copy into SCiLS.

**Input**: Did you run the [OPTIONAL] block of code right before this?
*   If you did, **no input.** Just press play.
*   If you didn't, upload the **One (1) METASPACE .CSV file** you just downloaded here.



In [ ]:
import sys
import os
import traceback
import numpy as np
import pandas as pd
from google.colab import files

# Checks to see if you already uploaded files in the previous optional step; if not, it skips this.
if os.path.exists("Merged.csv"):
  filename="Merged.csv"

# Grabs the file that you upload and stores the name
else:
  userinfile = files.upload()
  filename = list(userinfile.keys())[0]

# Asking for the raw Off percent threshold
rawpercent = 1.0
rawpercent = float(input("What raw percent threshold do you want? (i.e. 0.2) "))

# Figuring out whether to read the merged file or the new one that got uploaded in this block
if "Merged" in filename:
  infile = mergedfile
else:
  infile = pd.read_csv(filename, skiprows=2)

# Takes out all rows with duplicate m.z. values
notdupes = infile.drop_duplicates(subset=["mz"])

# Conserves only the rows that are under the rawOff precent threshold (or has no value in that column)
noraw = notdupes[(notdupes["rawOffSampleProb"] < rawpercent) | (notdupes["rawOffSampleProb"].isna())]

# Exports this to a .csv, which you can double click to open the file in a window in colab, then get the SCiLS m.z. list you need. Copy and pasteable!
noraw["mz"].to_csv("m.z. values to put into SCiLS.csv", header=None, index=False)

**Purpose**: Takes your SCiLS output to annotate and reformat the file, and then outputs an xslx file ready for fold change analysis.

**Input**: One(1) .CSV file from SCiLS.

In [ ]:
from typing_extensions import final
from datetime import datetime

# Grabs the file from SCiLS
scilsinfile = files.upload()
scilsfilename = list(scilsinfile.keys())[0]

# Reads in the file, and drops some of the column names
infilescils = pd.read_csv(scilsfilename, sep=";", skiprows=8)
infilescils = infilescils.drop(columns=['Interval Width (+/- Da)', 'Color', 'Name'])

#Prepares to align both dataframes by mz values for merging (since one file goes to 5 decimal places and the other only to 4)
noraw['mz'] = noraw['mz'].round(4)

# Merges the file via mz values
finaldf = infilescils.join(noraw[['moleculeNames', 'ion', 'adduct', 'datasetName']], how='inner')

# Reorganizes the file so that the names for the adducts, etc. are at the end of the file and the m.z is at the front
cols = finaldf.columns.tolist()
cols = cols[-4:] + cols[:-4]
finaldf = finaldf[cols]

# Gets today's date to add to the file name
outfiledate = datetime.today().strftime('%Y%m%d')

# Names the file for output for analysis handoff
if "Merged" in filename:
  outfilename = f"{outfiledate}_Merged_For_Analysis_{mf1name.removesuffix(".csv")}.xlsx"
else:
  outfilename = f"{outfiledate}_For_Analysis_{filename.removesuffix(".csv")}.xlsx"

# Exports the final excel file for handoff so that you can download it
finaldf.to_excel(outfilename, index=False)
